# Sprint 4 · US 4.1 — frozen wav2vec 2.0 / HuBERT / WavLM: embedding extraction (sections 1–9) and task heads (sections 10–14)

All logic lives in the repo (`src/features/ssl_embeddings.py`, `src/models/frozen_heads.py`); this notebook only sets up Colab and calls it.
Run top to bottom. **After a disconnect:** re-run sections 1–6, then the cell you were on — finished shards / jobs are skipped.
The embedding cache lives on local Colab disk, so run sections 10–14 in the same runtime as section 8 (or re-run section 8, it is resumable).

Before the first run: (1) `python experiments/scripts/pack_colab_data.py` on your machine, (2) upload `dist/esl_colab_pack.tar` to
`MyDrive/esl-pronunciation-data/`, (3) `git push` the Sprint 4 code. Runtime → Change runtime type → **T4 GPU**.
For sections 10+: `python experiments/scripts/export_side_features.py && python experiments/scripts/pack_colab_data.py --side`, then upload `dist/esl_side_pack.tar` the same way.


## 1 · GPU check

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(out or "NO GPU - Runtime > Change runtime type > T4 GPU (extraction still works on CPU but is ~20x slower)")

## 2 · Paths and Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

REPO_URL  = "https://github.com/christiannngg/esl-pronunciation-assessment.git"
PACK      = "/content/drive/MyDrive/esl-pronunciation-data/esl_colab_pack.tar"
DATA_ROOT = "/content/esl-data"                  # unpacked processed data (local Colab disk = fast)
CACHE     = "/content/ssl_cache"                 # embedding cache, ~6 GB per encoder, disposable (re-extraction takes minutes)
OUT       = "/content/drive/MyDrive/esl-pronunciation-data/frozen_outputs"   # small results only (manifests, tables, predictions)
os.makedirs(OUT, exist_ok=True)
os.environ["DATA_ROOT"] = DATA_ROOT
os.environ["SSL_CACHE_ROOT"] = CACHE

## 3 · Get the code (private repo: add a Colab secret named `GITHUB_TOKEN`)

In [ ]:
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass                                         # public repo or no secret: plain clone

os.chdir("/content")
if not os.path.exists("esl-pronunciation-assessment"):
    subprocess.run(["git", "clone", "--quiet", url], check=True)
os.chdir("/content/esl-pronunciation-assessment")
subprocess.run(["git", "pull", "--ff-only", "--quiet"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 4 · Dependencies (Colab already ships torch and transformers; versions are recorded in each manifest)

In [ ]:
!pip -q install soundfile pyyaml tqdm
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

## 5 · Unpack the data pack onto local disk

In [ ]:
import shutil, subprocess, os
LOCAL_PACK = "/content/esl_colab_pack.tar"
DONE = f"{DATA_ROOT}/.unpacked"
if not os.path.exists(DONE):
    print("pack on Drive:", round(os.path.getsize(PACK) / 1e9, 3), "GB")     # FileNotFoundError here = wrong PACK path (see section 2)
    shutil.copyfile(PACK, LOCAL_PACK)                                       # local copy first: more reliable than reading a 1 GB tar over the Drive mount
    print("copied to local disk:", round(os.path.getsize(LOCAL_PACK) / 1e9, 3), "GB")
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    os.makedirs(DATA_ROOT)
    r = subprocess.run(["tar", "-xf", LOCAL_PACK, "-C", DATA_ROOT], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError("tar failed (the archive is probably truncated - re-upload it):\n" + r.stderr[-2000:])
    open(DONE, "w").write("ok")
    os.remove(LOCAL_PACK)
print(open(f"{DATA_ROOT}/PACK_MANIFEST.json").read())

## 6 · Dry run + unit tests (no GPU needed; validates every Sprint 3 row resolves to one pooling span)

In [ ]:
!python -m src.features.ssl_embeddings --dry-run
!pip -q install pytest
!python -m pytest tests/test_ssl_embeddings.py -q

## 7 · Smoke test — 20 utterances per dataset, one encoder (about a minute, mostly model download)

In [ ]:
!python -m src.features.ssl_embeddings --encoder wav2vec2_base --limit 20 --overwrite
!python -m src.features.ssl_embeddings --encoder wav2vec2_base --limit 20 --verify

## 8 · Full extraction — one cell per encoder (resumable)

In [ ]:
!python -m src.features.ssl_embeddings --encoder wav2vec2_base
!python -m src.features.ssl_embeddings --encoder wav2vec2_base --verify

In [ ]:
!python -m src.features.ssl_embeddings --encoder hubert_base
!python -m src.features.ssl_embeddings --encoder hubert_base --verify

In [ ]:
!python -m src.features.ssl_embeddings --encoder wavlm_base_plus
!python -m src.features.ssl_embeddings --encoder wavlm_base_plus --verify

## 9 · Keep the manifests (tiny) on Drive

In [ ]:
import glob, shutil
for m in glob.glob(f"{CACHE}/*/manifest.json"):
    name = m.split("/")[-2]
    shutil.copy(m, f"{OUT}/manifest_{name}.json")
    print("saved", name)
!du -sh {CACHE}/*

## 10 · Side features for the heads (small second pack: canonical phone one-hot, durations, word position - the same non-acoustic columns the MFCC baselines had)

In [ ]:
import os, shutil, subprocess
SIDE_PACK = "/content/drive/MyDrive/esl-pronunciation-data/esl_side_pack.tar"     # or wherever you uploaded it
if not os.path.exists(SIDE_PACK):
    SIDE_PACK = "/content/esl-pronunciation-assessment/esl_side_pack.tar"
r = subprocess.run(["tar", "-xf", SIDE_PACK, "-C", DATA_ROOT], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("could not unpack the side pack (set SIDE_PACK to its path):\n" + r.stderr[-1500:])
print(sorted(f for f in os.listdir(f"{DATA_ROOT}/processed/features/tables") if "side" in f))

## 11 · Tests for the heads (includes the real torch trainer: it must learn a toy regression / classification, be seed-deterministic, early-stop)

In [ ]:
!python -m pytest tests/test_frozen_heads.py -q

## 12 · Smoke run — two cheap tasks, one encoder (about a minute); the printed score is validation RMSE (regression) or macro-F1 (classification)

In [ ]:
!python -m src.models.frozen_heads run --encoder wav2vec2_base --stage search --tasks sent_accuracy l2_word_error --layer 12

## 13 · Full head runs, one cell per encoder, layer 12 (`search` = every grid point with one seed, `final` = selected configs with 3 seeds). Resumable; `--budget 3000` stops cleanly before the Colab limit.

In [ ]:
for enc in ["wav2vec2_base"]:
    !python -m src.models.frozen_heads run --encoder {enc} --stage search --budget 3000
    !python -m src.models.frozen_heads run --encoder {enc} --stage final  --budget 3000
    !python -m src.models.frozen_heads status --encoder {enc}

In [ ]:
for enc in ["hubert_base"]:
    !python -m src.models.frozen_heads run --encoder {enc} --stage search --budget 3000
    !python -m src.models.frozen_heads run --encoder {enc} --stage final  --budget 3000
    !python -m src.models.frozen_heads status --encoder {enc}

In [ ]:
for enc in ["wavlm_base_plus"]:
    !python -m src.models.frozen_heads run --encoder {enc} --stage search --budget 3000
    !python -m src.models.frozen_heads run --encoder {enc} --stage final  --budget 3000
    !python -m src.models.frozen_heads status --encoder {enc}

## 14 · Copy the head predictions (a few MB) to Drive, then download `frozen_results` into `experiments/results/frozen/` on your machine and run `python -m src.evaluation.report_frozen --encoders wavlm_base_plus hubert_base wav2vec2_base`

In [ ]:
import shutil
shutil.copytree("/content/esl-pronunciation-assessment/experiments/results/frozen", f"{OUT}/frozen_results", dirs_exist_ok=True)
!du -sh {OUT}/frozen_results

## 15 · US 4.2 runs — pull the new code first (re-run the section 3 cell), then run these. Everything is resumable; re-run section 14 afterwards to copy the results to Drive.

### 15a · Layer sweep: one linear probe per layer 0–12 and task (layer 12 is already done by section 13). About 5 min per layer on a T4; use `--layers 4-10` for a shorter first pass.

In [ ]:
for enc in ["wavlm_base_plus"]:
    !python -m src.models.frozen_heads run --encoder {enc} --probe --layers 0-12 --budget 3000

In [ ]:
for enc in ["hubert_base"]:
    !python -m src.models.frozen_heads run --encoder {enc} --probe --layers 0-12 --budget 3000

In [ ]:
for enc in ["wav2vec2_base"]:
    !python -m src.models.frozen_heads run --encoder {enc} --probe --layers 0-12 --budget 3000

### 15b · Input ablations for WavLM (main setting = embedding + side columns is already done). `side` uses no audio embedding, so the encoder name only labels the folder.

In [ ]:
for mode in ["emb", "side"]:
    !python -m src.models.frozen_heads run --encoder wavlm_base_plus --mode {mode} --stage search --budget 3000
    !python -m src.models.frozen_heads run --encoder wavlm_base_plus --mode {mode} --stage final  --budget 3000

### 15c · Re-fit the classification `final` files so they also store train probabilities (fixes the train-score column of the `cw+thr` rows). Only the three classification tasks, only WavLM and HuBERT.

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base"]:
    !python -m src.models.frozen_heads run --encoder {enc} --stage final --force --tasks phone_mispron_t1.0 l2_phone_error l2_word_error --budget 3000

## 16 · US 4.3 — fixes (layer re-selection, learned layer-weighted sum, sentence regularisation, word pooling, child/teen thresholds)
Everything is resumable. Before the first run on a fresh runtime: (1) on your machine `python experiments/scripts/pack_colab_data.py --pros` and upload `dist/esl_pros_pack.tar` next to the other packs, (2) `git push` the Sprint 4.3 code, (3) re-run **sections 1–6** (GPU check, paths, code pull, dependencies, data pack) and **section 10** (side features).
The embedding cache from section 8 lives on local Colab disk and is gone after a runtime reset, so 16a re-extracts it (the extractor now also writes the `so762_word_ms` unit: max and std pooling of each word's frames, used by the word-accuracy fix).
Run order: 16.0 → 16a → 16b → 16c → 16d → 16e → 16f → 16g → 16h. Each head call takes `--budget 3000` seconds and stops cleanly; re-run the same cell to continue.

### 16.0 · Prosody columns and the earlier results (the layer probes and the layer-12 benchmark are the inputs of fix 1 and fix 5)

In [ ]:
import os, shutil, subprocess
PROS_PACK = "/content/drive/MyDrive/esl-pronunciation-data/esl_pros_pack.tar"
r = subprocess.run(["tar", "-xf", PROS_PACK, "-C", DATA_ROOT], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("could not unpack the prosody pack (set PROS_PACK to its path):\n" + r.stderr[-1500:])
print(sorted(f for f in os.listdir(f"{DATA_ROOT}/processed/features/tables") if "pros" in f))

REPO_DIR = "/content/esl-pronunciation-assessment"
shutil.copytree(f"{OUT}/frozen_results", f"{REPO_DIR}/experiments/results/frozen", dirs_exist_ok=True)      # section 14/15 outputs saved earlier
!du -sh {REPO_DIR}/experiments/results/frozen
!ls {REPO_DIR}/experiments/results/frozen/wavlm_base_plus

### 16a · Re-extract the embedding cache (about 6 GB per encoder + the small word max/std unit). Skip an encoder whose cache is still on disk: the check below says so.

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base", "wav2vec2_base"]:
    !python -m src.features.ssl_embeddings --encoder {enc}
    !python -m src.features.ssl_embeddings --encoder {enc} --verify

### 16b · Layer probes for the three sentence tasks that had none (fix 1 ranks layers 6–12 per task by probe validation score; the other six tasks were probed in 15a). About a minute per layer.

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base", "wav2vec2_base"]:
    !python -m src.models.frozen_heads run --encoder {enc} --probe --layers 6-12 --tasks sent_fluency sent_prosodic sent_total --budget 3000

### 16c · Fix 1 · layer re-selection (layers 6–12, per task): print the probe-ranked shortlist first, then run the full head protocol (both heads, 3 seeds) on the shortlisted layers. Layers already fitted in section 13 are skipped.

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base", "wav2vec2_base"]:
    !python -m src.models.frozen_fix layers --encoder {enc} --dry-run

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base", "wav2vec2_base"]:
    !python -m src.models.frozen_fix layers --encoder {enc} --budget 3000

### 16d · Fix 2 · learned layer-weighted sum over layers 0–12, all nine tasks, same grids and 3 seeds as US 4.1 (so the delta isolates the weighted sum). The `final` stage re-fits the validation-selected configs.

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base", "wav2vec2_base"]:
    !python -m src.models.frozen_fix run --encoder {enc} --recipe ws --stage search --budget 3000
    !python -m src.models.frozen_fix run --encoder {enc} --recipe ws --stage final  --budget 3000
    !python -m src.models.frozen_fix status --encoder {enc} --recipe ws

### 16e · Fix 3 · sentence regularisation (stronger weight decay / dropout / input dropout, PCA on the layer-averaged input, 5-seed averaging), with and without the US 3.3 pitch / energy / pause columns. Small tables: a few minutes per encoder.

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base", "wav2vec2_base"]:
    for recipe in ["sent_ws", "sent_ws_pros", "sent_avgpca", "sent_avgpca_pros"]:
        !python -m src.models.frozen_fix run --encoder {enc} --recipe {recipe} --stage search --budget 3000
        !python -m src.models.frozen_fix run --encoder {enc} --recipe {recipe} --stage final  --budget 3000

### 16f · Fix 4 · word accuracy: pooling over the word's frames (mean = control, max, mean+max, mean+max+std), all with the layer-weighted sum and the prosody columns.

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base", "wav2vec2_base"]:
    for recipe in ["word_mean_pros", "word_max_pros", "word_meanmax_pros", "word_meanmaxstd_pros"]:
        !python -m src.models.frozen_fix run --encoder {enc} --recipe {recipe} --stage search --budget 3000
        !python -m src.models.frozen_fix run --encoder {enc} --recipe {recipe} --stage final  --budget 3000

### 16g · Fix 5 · child/teen mispronunciation: out-of-fold probabilities of the validation-best class-weighted head (speaker-grouped, train speakers only). Needs 16c and 16d first, because the source head is chosen among layers 6–12 and the weighted sum.

In [ ]:
for enc in ["wavlm_base_plus", "hubert_base", "wav2vec2_base"]:
    !python -m src.models.frozen_fix mispron --encoder {enc}

### 16h · Optional · refill the train-score column of wav2vec2's `cw+thr` rows (same as 15c, which only covered WavLM and HuBERT)

In [ ]:
!python -m src.models.frozen_heads run --encoder wav2vec2_base --stage final --force --tasks phone_mispron_t1.0 l2_phone_error l2_word_error --budget 3000

### 16i · Copy the results to Drive, then download `frozen_results` and `frozen43` into `experiments/results/` on your machine and run `python -m src.evaluation.report_us43 --encoders wavlm_base_plus hubert_base wav2vec2_base`

In [ ]:
import shutil
shutil.copytree(f"{REPO_DIR}/experiments/results/frozen", f"{OUT}/frozen_results", dirs_exist_ok=True)
shutil.copytree(f"{REPO_DIR}/experiments/results/frozen43", f"{OUT}/frozen43", dirs_exist_ok=True)
!du -sh {OUT}/frozen_results {OUT}/frozen43